In [ ]:
# HW10-11 — компьютерное зрение в PyTorch
# Part A: classification on STL10
# Part B: segmentation on Pascal VOC with DeepLabV3_ResNet50

import os
import json
import csv
import copy
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import STL10, VOCSegmentation
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights


# =========================================================
# 1. Seed, device, paths
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "homeworks").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())
HW_DIR = REPO_ROOT / "homeworks" / "HW10-11"
ART_DIR = HW_DIR / "artifacts"
FIG_DIR = ART_DIR / "figures"
DATA_DIR = REPO_ROOT / "data"

HW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("HW_DIR:", HW_DIR)
print("ART_DIR:", ART_DIR)
print("FIG_DIR:", FIG_DIR)
print("DATA_DIR:", DATA_DIR)


# =========================================================
# 2. Config
# =========================================================
CLASSIFICATION_DATASET = "STL10"
STRUCTURED_DATASET = "Pascal VOC"
STRUCTURED_TRACK = "segmentation"

CLASS_BATCH_SIZE_CNN = 64 if device.type == "cuda" else 64
CLASS_BATCH_SIZE_RESNET = 32 if device.type == "cuda" else 32

C1_EPOCHS = 6
C2_EPOCHS = 6
C3_EPOCHS = 4
C4_EPOCHS = 4

SEG_INPUT_SIZE = (320, 320)
SEG_EVAL_LIMIT = 40
VOC_PERSON_CLASS = 15

RUNS_CSV_PATH = ART_DIR / "runs.csv"
BEST_CLASSIFIER_PATH = ART_DIR / "best_classifier.pt"
BEST_CLASSIFIER_CONFIG_PATH = ART_DIR / "best_classifier_config.json"

CLASSIFICATION_CURVES_PATH = FIG_DIR / "classification_curves_best.png"
CLASSIFICATION_COMPARE_PATH = FIG_DIR / "classification_compare.png"
AUGMENTATIONS_PREVIEW_PATH = FIG_DIR / "augmentations_preview.png"
SEGMENTATION_EXAMPLES_PATH = FIG_DIR / "segmentation_examples.png"
SEGMENTATION_METRICS_PATH = FIG_DIR / "segmentation_metrics.png"


# =========================================================
# 3. Helpers
# =========================================================
def denormalize_tensor(x: torch.Tensor, mean, std):
    mean_t = torch.tensor(mean, dtype=x.dtype, device=x.device).view(-1, 1, 1)
    std_t = torch.tensor(std, dtype=x.dtype, device=x.device).view(-1, 1, 1)
    return x * std_t + mean_t

def show_tensor_image(ax, img_tensor, mean, std, title=""):
    img = denormalize_tensor(img_tensor.cpu(), mean, std).clamp(0, 1)
    img = img.permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

class TransformSubset(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, target = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, target

def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

def write_runs_csv(rows, path: Path):
    fields = [
        "experiment_id", "task", "dataset", "seed", "model_summary", "optimizer",
        "lr", "epochs_trained", "best_val_accuracy", "test_accuracy",
        "precision", "recall", "mean_iou", "notes"
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


# =========================================================
# 4. Part A: STL10 classification
# =========================================================
print("\n=== PART A: CLASSIFICATION ===")

# Raw datasets without transforms
stl_train_raw = STL10(root=str(DATA_DIR), split="train", download=True, transform=None)
stl_test_raw = STL10(root=str(DATA_DIR), split="test", download=True, transform=None)

class_names = stl_train_raw.classes
num_classes = len(class_names)

# Fixed train/val split
n_total = len(stl_train_raw)
n_val = int(0.2 * n_total)
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n_total, generator=g).tolist()
val_indices = perm[:n_val]
train_indices = perm[n_val:]

print("STL10 sizes:", n_train, n_val, len(stl_test_raw))
print("Num classes:", num_classes)

# Transforms
cnn_mean = (0.5, 0.5, 0.5)
cnn_std = (0.5, 0.5, 0.5)

cnn_base_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(cnn_mean, cnn_std),
])

cnn_aug_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(96, padding=8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(cnn_mean, cnn_std),
])

resnet_weights = ResNet18_Weights.DEFAULT
resnet_preprocess = resnet_weights.transforms()

# Datasets per experiment
train_c1 = TransformSubset(stl_train_raw, train_indices, transform=cnn_base_transform)
val_c1 = TransformSubset(stl_train_raw, val_indices, transform=cnn_base_transform)
test_c1 = TransformSubset(stl_test_raw, range(len(stl_test_raw)), transform=cnn_base_transform)

train_c2 = TransformSubset(stl_train_raw, train_indices, transform=cnn_aug_transform)
val_c2 = TransformSubset(stl_train_raw, val_indices, transform=cnn_base_transform)
test_c2 = TransformSubset(stl_test_raw, range(len(stl_test_raw)), transform=cnn_base_transform)

train_c3 = TransformSubset(stl_train_raw, train_indices, transform=resnet_preprocess)
val_c3 = TransformSubset(stl_train_raw, val_indices, transform=resnet_preprocess)
test_c3 = TransformSubset(stl_test_raw, range(len(stl_test_raw)), transform=resnet_preprocess)

train_c4 = TransformSubset(stl_train_raw, train_indices, transform=resnet_preprocess)
val_c4 = TransformSubset(stl_train_raw, val_indices, transform=resnet_preprocess)
test_c4 = TransformSubset(stl_test_raw, range(len(stl_test_raw)), transform=resnet_preprocess)

# Sanity check
sanity_loader = DataLoader(train_c1, batch_size=CLASS_BATCH_SIZE_CNN, shuffle=True)
x, y = next(iter(sanity_loader))
print("Classification sanity batch x:", x.shape)
print("Classification sanity batch y:", y.shape)

# Preview examples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_c1[i]
    show_tensor_image(ax, img, cnn_mean, cnn_std, title=class_names[label])
fig.suptitle("STL10 examples")
plt.tight_layout()
plt.show()

# Augmentations preview
raw_img, raw_label = stl_train_raw[0]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(raw_img)
axes[0].set_title("original")
axes[0].axis("off")
for i in range(1, 5):
    aug_img = cnn_aug_transform(raw_img)
    show_tensor_image(axes[i], aug_img, cnn_mean, cnn_std, title=f"aug {i}")
fig.suptitle(f"Augmentations preview ({class_names[raw_label]})")
plt.tight_layout()
fig.savefig(AUGMENTATIONS_PREVIEW_PATH, dpi=150)
plt.show()
plt.close(fig)


# ---------------------------------------------------------
# Models
# ---------------------------------------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def build_simple_cnn(num_classes: int):
    return SimpleCNN(num_classes)

def build_resnet18_head_only(num_classes: int):
    model = resnet18(weights=resnet_weights)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18_finetune(num_classes: int):
    model = resnet18(weights=resnet_weights)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@dataclass
class ClfExperimentResult:
    experiment_id: str
    model_summary: str
    optimizer_name: str
    lr: float
    epochs_trained: int
    best_val_accuracy: float
    history: Dict[str, list]
    best_state: Dict[str, torch.Tensor]
    build_name: str
    batch_size: int
    train_transform_name: str
    eval_transform_name: str
    notes: str

def run_classification_experiment(
    experiment_id: str,
    model,
    train_ds,
    val_ds,
    batch_size: int,
    epochs: int,
    lr: float,
    optimizer_name: str,
    model_summary: str,
    build_name: str,
    train_transform_name: str,
    eval_transform_name: str,
    notes: str,
):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    else:
        raise ValueError("Only Adam is used in this notebook.")

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"{experiment_id} | epoch {epoch}/{epochs} | train_acc={tr_acc:.4f} | val_acc={va_acc:.4f}")

    return ClfExperimentResult(
        experiment_id=experiment_id,
        model_summary=model_summary,
        optimizer_name=optimizer_name,
        lr=lr,
        epochs_trained=epochs,
        best_val_accuracy=best_val_acc,
        history=history,
        best_state=best_state,
        build_name=build_name,
        batch_size=batch_size,
        train_transform_name=train_transform_name,
        eval_transform_name=eval_transform_name,
        notes=notes,
    )

# Run C1-C4
runs_rows = []

clf_results: Dict[str, ClfExperimentResult] = {}

C1_model = build_simple_cnn(num_classes)
print("C1 trainable params:", count_trainable_params(C1_model))
clf_results["C1"] = run_classification_experiment(
    experiment_id="C1",
    model=C1_model,
    train_ds=train_c1,
    val_ds=val_c1,
    batch_size=CLASS_BATCH_SIZE_CNN,
    epochs=C1_EPOCHS,
    lr=1e-3,
    optimizer_name="Adam",
    model_summary="SimpleCNN, no augmentations",
    build_name="simple_cnn",
    train_transform_name="cnn_base_transform",
    eval_transform_name="cnn_base_transform",
    notes="simple-cnn-base"
)

C2_model = build_simple_cnn(num_classes)
print("C2 trainable params:", count_trainable_params(C2_model))
clf_results["C2"] = run_classification_experiment(
    experiment_id="C2",
    model=C2_model,
    train_ds=train_c2,
    val_ds=val_c2,
    batch_size=CLASS_BATCH_SIZE_CNN,
    epochs=C2_EPOCHS,
    lr=1e-3,
    optimizer_name="Adam",
    model_summary="SimpleCNN, augmentations",
    build_name="simple_cnn",
    train_transform_name="cnn_aug_transform",
    eval_transform_name="cnn_base_transform",
    notes="simple-cnn-aug"
)

C3_model = build_resnet18_head_only(num_classes)
print("C3 trainable params:", count_trainable_params(C3_model))
clf_results["C3"] = run_classification_experiment(
    experiment_id="C3",
    model=C3_model,
    train_ds=train_c3,
    val_ds=val_c3,
    batch_size=CLASS_BATCH_SIZE_RESNET,
    epochs=C3_EPOCHS,
    lr=1e-3,
    optimizer_name="Adam",
    model_summary="ResNet18 pretrained, head-only",
    build_name="resnet18_head_only",
    train_transform_name="resnet_preprocess",
    eval_transform_name="resnet_preprocess",
    notes="resnet18-head-only"
)

C4_model = build_resnet18_finetune(num_classes)
print("C4 trainable params:", count_trainable_params(C4_model))
clf_results["C4"] = run_classification_experiment(
    experiment_id="C4",
    model=C4_model,
    train_ds=train_c4,
    val_ds=val_c4,
    batch_size=CLASS_BATCH_SIZE_RESNET,
    epochs=C4_EPOCHS,
    lr=1e-4,
    optimizer_name="Adam",
    model_summary="ResNet18 pretrained, layer4+fc fine-tune",
    build_name="resnet18_finetune",
    train_transform_name="resnet_preprocess",
    eval_transform_name="resnet_preprocess",
    notes="resnet18-finetune"
)

# Fill rows for C1-C4 (test_accuracy filled only later for best)
for exp_id, res in clf_results.items():
    runs_rows.append({
        "experiment_id": exp_id,
        "task": "classification",
        "dataset": CLASSIFICATION_DATASET,
        "seed": SEED,
        "model_summary": res.model_summary,
        "optimizer": res.optimizer_name,
        "lr": res.lr,
        "epochs_trained": res.epochs_trained,
        "best_val_accuracy": res.best_val_accuracy,
        "test_accuracy": "",
        "precision": "",
        "recall": "",
        "mean_iou": "",
        "notes": res.notes
    })

# Best classifier by val accuracy
best_clf_id = max(clf_results.keys(), key=lambda k: clf_results[k].best_val_accuracy)
best_clf_result = clf_results[best_clf_id]
print("Best classification experiment:", best_clf_id, best_clf_result.best_val_accuracy)

# Save best classifier weights
torch.save(best_clf_result.best_state, BEST_CLASSIFIER_PATH)

# Rebuild model and evaluate on test ONCE
if best_clf_result.build_name == "simple_cnn":
    best_model = build_simple_cnn(num_classes)
    best_test_ds = test_c1
elif best_clf_result.build_name == "resnet18_head_only":
    best_model = build_resnet18_head_only(num_classes)
    best_test_ds = test_c3
elif best_clf_result.build_name == "resnet18_finetune":
    best_model = build_resnet18_finetune(num_classes)
    best_test_ds = test_c4
else:
    raise ValueError("Unknown build_name")

best_model.load_state_dict(torch.load(BEST_CLASSIFIER_PATH, map_location="cpu"))
best_model = best_model.to(device)

best_test_loader = DataLoader(best_test_ds, batch_size=best_clf_result.batch_size, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))
criterion = nn.CrossEntropyLoss()
best_test_loss, best_test_acc = evaluate(best_model, best_test_loader, criterion, device)

print("Final classification TEST accuracy:", best_test_acc)

# Update best row with test_accuracy
for row in runs_rows:
    if row["experiment_id"] == best_clf_id:
        row["test_accuracy"] = best_test_acc

# Save best_classifier_config.json
best_classifier_config = {
    "experiment_id": best_clf_id,
    "dataset": CLASSIFICATION_DATASET,
    "seed": SEED,
    "device": str(device),

    "architecture": best_clf_result.model_summary,
    "model_summary": best_clf_result.model_summary,
    "optimizer": best_clf_result.optimizer_name,
    "lr": best_clf_result.lr,
    "batch_size": best_clf_result.batch_size,
    "epochs_trained": best_clf_result.epochs_trained,
    "best_val_accuracy": best_clf_result.best_val_accuracy,
    "test_accuracy": best_test_acc,

    "train_transform": best_clf_result.train_transform_name,
    "eval_transform": best_clf_result.eval_transform_name,
    "num_classes": num_classes,
    "train_size": n_train,
    "val_size": n_val,
    "test_size": len(stl_test_raw),

    "data": {
        "dataset": CLASSIFICATION_DATASET,
        "num_classes": num_classes,
        "train_size": n_train,
        "val_size": n_val,
        "test_size": len(stl_test_raw),
    },
    "model": {
        "build_name": best_clf_result.build_name,
        "model_summary": best_clf_result.model_summary,
    },
    "training": {
        "criterion": "CrossEntropyLoss",
        "optimizer": best_clf_result.optimizer_name,
        "lr": best_clf_result.lr,
        "batch_size": best_clf_result.batch_size,
        "epochs_trained": best_clf_result.epochs_trained,
        "criterion_selection": "best_val_accuracy",
    },
    "transforms": {
        "train_transform": best_clf_result.train_transform_name,
        "eval_transform": best_clf_result.eval_transform_name,
    },
    "metrics": {
        "best_val_accuracy": best_clf_result.best_val_accuracy,
        "test_accuracy": best_test_acc,
    }
}

with open(BEST_CLASSIFIER_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(best_classifier_config, f, ensure_ascii=False, indent=2)

print("Saved:", BEST_CLASSIFIER_PATH)
print("Saved:", BEST_CLASSIFIER_CONFIG_PATH)

# Plot best classification curves
best_hist = best_clf_result.history
epochs_arr = np.arange(1, len(best_hist["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_arr, best_hist["train_loss"], label="train_loss")
axes[0].plot(epochs_arr, best_hist["val_loss"], label="val_loss")
axes[0].set_title(f"{best_clf_id} loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs_arr, best_hist["train_acc"], label="train_acc")
axes[1].plot(epochs_arr, best_hist["val_acc"], label="val_acc")
axes[1].set_title(f"{best_clf_id} accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(CLASSIFICATION_CURVES_PATH, dpi=150)
plt.show()
plt.close(fig)

# Plot C1-C4 comparison
compare_ids = ["C1", "C2", "C3", "C4"]
compare_vals = [clf_results[k].best_val_accuracy for k in compare_ids]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(compare_ids, compare_vals)
ax.set_title("Classification comparison: best val accuracy")
ax.set_ylabel("best_val_accuracy")
plt.tight_layout()
plt.savefig(CLASSIFICATION_COMPARE_PATH, dpi=150)
plt.show()
plt.close(fig)


# =========================================================
# 5. Part B: Pascal VOC segmentation
# =========================================================
print("\n=== PART B: SEGMENTATION ===")

voc_val = VOCSegmentation(root=str(DATA_DIR), year="2012", image_set="val", download=True)

VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor"
]

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg_mean = (0.485, 0.456, 0.406)
seg_std = (0.229, 0.224, 0.225)

seg_image_transform = transforms.Compose([
    transforms.Resize(SEG_INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(seg_mean, seg_std),
])

def resize_mask_nearest(mask_pil: Image.Image, size_hw: Tuple[int, int]):
    h, w = size_hw
    mask_resized = mask_pil.resize((w, h), resample=Image.NEAREST)
    return np.array(mask_resized, dtype=np.uint8)

# Find first images that contain "person" in GT
def collect_indices_with_class(dataset, class_idx: int, limit: int):
    found = []
    for i in range(len(dataset)):
        _, mask_pil = dataset[i]
        mask = np.array(mask_pil, dtype=np.uint8)
        if (mask == class_idx).any():
            found.append(i)
            if len(found) >= limit:
                break
    return found

seg_eval_indices = collect_indices_with_class(voc_val, VOC_PERSON_CLASS, SEG_EVAL_LIMIT)
print("VOC val subset for segmentation metrics:", len(seg_eval_indices))

# Sanity-check examples
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, idx in zip(axes.flat, seg_eval_indices[:6]):
    img_pil, mask_pil = voc_val[idx]
    ax.imshow(img_pil)
    ax.set_title(f"idx={idx}")
    ax.axis("off")
plt.suptitle("VOC segmentation sanity-check examples")
plt.tight_layout()
plt.show()

# Model
seg_model = deeplabv3_resnet50(weights=seg_weights).to(device)
seg_model.eval()

def binary_opening(mask_bool: torch.Tensor) -> torch.Tensor:
    # mask_bool: H x W, bool
    x = mask_bool.float().unsqueeze(0).unsqueeze(0)
    eroded = 1.0 - F.max_pool2d(1.0 - x, kernel_size=3, stride=1, padding=1)
    opened = F.max_pool2d(eroded, kernel_size=3, stride=1, padding=1)
    return (opened[0, 0] > 0.5)

def binary_metrics(pred_mask: torch.Tensor, gt_mask: torch.Tensor, valid_mask: torch.Tensor):
    pred = pred_mask[valid_mask]
    gt = gt_mask[valid_mask]

    tp = torch.logical_and(pred, gt).sum().item()
    fp = torch.logical_and(pred, ~gt).sum().item()
    fn = torch.logical_and(~pred, gt).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)

    return tp, fp, fn, precision, recall, iou

def make_overlay(image_pil: Image.Image, mask_bool: torch.Tensor, color=(255, 0, 0), alpha=0.4):
    img = np.array(image_pil.resize((SEG_INPUT_SIZE[1], SEG_INPUT_SIZE[0])), dtype=np.float32)
    mask = mask_bool.cpu().numpy().astype(bool)

    overlay = img.copy()
    overlay[mask] = (1 - alpha) * overlay[mask] + alpha * np.array(color, dtype=np.float32)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay

@torch.no_grad()
def run_segmentation_mode(mode_name: str):
    total_tp, total_fp, total_fn = 0.0, 0.0, 0.0
    ious = []
    example_records = []

    for idx in seg_eval_indices:
        img_pil, mask_pil = voc_val[idx]

        img_tensor = seg_image_transform(img_pil).unsqueeze(0).to(device)
        logits = seg_model(img_tensor)["out"][0].cpu()
        pred_labels = logits.argmax(dim=0)

        gt_arr = torch.from_numpy(resize_mask_nearest(mask_pil, SEG_INPUT_SIZE))
        valid_mask = gt_arr != 255
        gt_fg = gt_arr == VOC_PERSON_CLASS

        pred_fg = pred_labels == VOC_PERSON_CLASS
        if mode_name == "V2":
            pred_fg = binary_opening(pred_fg)

        tp, fp, fn, precision_i, recall_i, iou_i = binary_metrics(pred_fg, gt_fg, valid_mask)

        total_tp += tp
        total_fp += fp
        total_fn += fn
        ious.append(iou_i)

        if len(example_records) < 3:
            example_records.append({
                "idx": idx,
                "image_pil": img_pil,
                "gt_fg": gt_fg,
                "pred_fg": pred_fg,
            })

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    mean_iou = float(np.mean(ious)) if len(ious) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "mean_iou": mean_iou,
        "examples": example_records,
    }

v1_metrics = run_segmentation_mode("V1")
v2_metrics = run_segmentation_mode("V2")

print("V1 metrics:", v1_metrics["precision"], v1_metrics["recall"], v1_metrics["mean_iou"])
print("V2 metrics:", v2_metrics["precision"], v2_metrics["recall"], v2_metrics["mean_iou"])

# Save segmentation_examples.png
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row_idx in range(3):
    rec_v1 = v1_metrics["examples"][row_idx]
    rec_v2 = v2_metrics["examples"][row_idx]

    img_pil = rec_v1["image_pil"]
    gt_overlay = make_overlay(img_pil, rec_v1["gt_fg"], color=(0, 255, 0), alpha=0.4)
    v1_overlay = make_overlay(img_pil, rec_v1["pred_fg"], color=(255, 0, 0), alpha=0.4)
    v2_overlay = make_overlay(img_pil, rec_v2["pred_fg"], color=(0, 0, 255), alpha=0.4)

    axes[row_idx, 0].imshow(img_pil.resize((SEG_INPUT_SIZE[1], SEG_INPUT_SIZE[0])))
    axes[row_idx, 0].set_title(f"image idx={rec_v1['idx']}")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(gt_overlay)
    axes[row_idx, 1].set_title("GT person")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(v1_overlay)
    axes[row_idx, 2].set_title("V1 prediction")
    axes[row_idx, 2].axis("off")

    axes[row_idx, 3].imshow(v2_overlay)
    axes[row_idx, 3].set_title("V2 prediction")
    axes[row_idx, 3].axis("off")

plt.tight_layout()
plt.savefig(SEGMENTATION_EXAMPLES_PATH, dpi=150)
plt.show()
plt.close(fig)

# Save segmentation_metrics.png
metric_names = ["precision", "recall", "mean_iou"]
v1_vals = [v1_metrics[k] for k in metric_names]
v2_vals = [v2_metrics[k] for k in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, v1_vals, width, label="V1")
ax.bar(x + width / 2, v2_vals, width, label="V2")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.0)
ax.set_title("Segmentation metrics")
ax.legend()
plt.tight_layout()
plt.savefig(SEGMENTATION_METRICS_PATH, dpi=150)
plt.show()
plt.close(fig)

# Add V1/V2 rows
runs_rows.append({
    "experiment_id": "V1",
    "task": "segmentation",
    "dataset": STRUCTURED_DATASET,
    "seed": SEED,
    "model_summary": "DeepLabV3_ResNet50 pretrained, foreground=person",
    "optimizer": "",
    "lr": "",
    "epochs_trained": "",
    "best_val_accuracy": "",
    "test_accuracy": "",
    "precision": v1_metrics["precision"],
    "recall": v1_metrics["recall"],
    "mean_iou": v1_metrics["mean_iou"],
    "notes": f"raw argmax mask; evaluated on first {len(seg_eval_indices)} val images containing person"
})

runs_rows.append({
    "experiment_id": "V2",
    "task": "segmentation",
    "dataset": STRUCTURED_DATASET,
    "seed": SEED,
    "model_summary": "DeepLabV3_ResNet50 pretrained, foreground=person",
    "optimizer": "",
    "lr": "",
    "epochs_trained": "",
    "best_val_accuracy": "",
    "test_accuracy": "",
    "precision": v2_metrics["precision"],
    "recall": v2_metrics["recall"],
    "mean_iou": v2_metrics["mean_iou"],
    "notes": f"morphological opening; evaluated on first {len(seg_eval_indices)} val images containing person"
})


# =========================================================
# 6. Save runs.csv + final summary
# =========================================================
write_runs_csv(runs_rows, RUNS_CSV_PATH)

print("\nSaved artifacts:")
print(" -", RUNS_CSV_PATH)
print(" -", BEST_CLASSIFIER_PATH)
print(" -", BEST_CLASSIFIER_CONFIG_PATH)
print(" -", CLASSIFICATION_CURVES_PATH)
print(" -", CLASSIFICATION_COMPARE_PATH)
print(" -", AUGMENTATIONS_PREVIEW_PATH)
print(" -", SEGMENTATION_EXAMPLES_PATH)
print(" -", SEGMENTATION_METRICS_PATH)

print("\nQuick summary for report.md:")
print("Best classification experiment:", best_clf_id)
print("Best val accuracy:", best_clf_result.best_val_accuracy)
print("Best test accuracy:", best_test_acc)
print("Segmentation V1:", v1_metrics)
print("Segmentation V2:", v2_metrics)